# SMS Spam Detection - Phase 1: exploration, tokenization and cleaning
Run the cells **top to bottom** (Kernel -> Restart & Run All). All logic lives in `src/`; this notebook calls it and shows the results (take your screenshots from here).

Prerequisites: `pip install -r requirements.txt` and `python -m src.setup_nltk`.

In [ ]:
import sys, re, json
from pathlib import Path
from collections import Counter
sys.path.insert(0, str(Path.cwd().parent))     # so `import src` works from notebooks/
import pandas as pd
import matplotlib.pyplot as plt
from src import config, load_data, explore, regex_patterns, cleaning
from src import tokenizers_compare as tc
pd.set_option("display.max_colwidth", 120)

## Task 1 - Dataset & environment

In [ ]:
df = load_data.load_raw()          # downloads to data/raw/ the first time (raw data is not committed to Git)
print(config.DATASET_NAME, "|", config.DATASET_PAGE, "|", config.DATASET_LICENCE)
df.head(10)

## Task 2 - Exploration & basic statistics

In [ ]:
stats = explore.basic_stats(df)
pd.Series(stats)

In [ ]:
sample = df.sample(n=min(config.SAMPLE_SIZE, len(df)), random_state=config.RANDOM_SEED)
v_sample, _ = explore.vocab_stats(sample["text"].tolist())
v_full, counter = explore.vocab_stats(df["text"].tolist())
print("SAMPLE:", {k: v for k, v in v_sample.items() if k != "top_words"})
print("FULL  :", {k: v for k, v in v_full.items() if k != "top_words"})
print("Top words:", v_full["top_words"])

In [ ]:
explore.class_and_length_plot(df, config.FIGURES / "class_length.png")
explore.zipf_plot(counter, config.FIGURES / "zipf.png")
from IPython.display import Image, display
display(Image(str(config.FIGURES / "class_length.png")), Image(str(config.FIGURES / "zipf.png")))

**Zipf / long tail - your comment:** *(write 2-3 sentences using the numbers above: share of hapax words, slope of the log-log plot, which words dominate).*

In [ ]:
for m in explore.find_messy_examples(df["text"].tolist()):
    print(f"[{m['kind']}]  {m['example']!r}\n")

## Task 3 - Tokenization & regular expressions

In [ ]:
demo = tc.compare_on_sentences(tc.pick_demo_sentences(df["text"].tolist()))
for d in demo:
    print("\nTEXT:", d["text"])
    for name in [k for k in d if k != "text"]:
        print(f"  {name:<10}", d[name])

In [ ]:
pd.DataFrame(tc.compare_on_corpus(df["text"].tolist()))

In [ ]:
pd.DataFrame(tc.challenge_counts(df["text"].tolist()))

In [ ]:
rows = regex_patterns.demo_counts(df["text"].tolist())
pd.DataFrame(rows)[["name", "n_messages", "examples", "pattern"]]

### Try a regex yourself
Change the pattern to see what it matches in the data.

In [ ]:
pattern = regex_patterns.MONEY
hits = df[df["text"].str.contains(pattern)].head(8)
for t in hits["text"]:
    print(pattern.findall(t), "<-", t[:100])

## Task 4 - Cleaning & normalization pipeline

In [ ]:
print(" -> ".join(cleaning.STEP_NAMES))
for name, _, why in cleaning.STEPS:
    print(f"- {name}: {why}")

In [ ]:
raw = df["text"].tolist()
steps = pd.DataFrame(cleaning.vocab_by_step(raw))
print(steps.to_string(index=False))
print("\nVOCAB BEFORE:", steps.iloc[0].vocab, " AFTER:", steps.iloc[-1].vocab)
explore.vocab_steps_plot(steps.to_dict("records"), config.FIGURES / "vocab_by_step.png")
steps.plot.bar(x="step", y="vocab", legend=False, figsize=(8, 3)); plt.show()

In [ ]:
df["text_clean"] = df["text"].map(cleaning.clean_text)      # the ONE cleaning function
df[["label", "text", "text_clean"]].sample(8, random_state=1)

In [ ]:
# Stemming vs lemmatization on frequent content words
words = Counter(t for x in raw for t in cleaning.apply_steps(x, "strip_punctuation").split()
                if t not in cleaning.get_stopwords() and t.isalpha() and len(t) > 2)
cmp = pd.DataFrame([{"word": w, "stem": cleaning.stem_word(w), "lemma": cleaning.lemmatize_word(w)}
                    for w, _ in words.most_common(300)])
cmp[cmp.stem != cmp.lemma].head(25)

In [ ]:
# Do we delete words that matter? Stop words most different between spam and ham
from nltk.corpus import stopwords
sw = set(stopwords.words("english"))
spam = df.label.eq("spam").tolist()
sp, hm = Counter(), Counter()
for x, s in zip(raw, spam):
    (sp if s else hm).update(set(cleaning.apply_steps(x, "strip_punctuation").split()) & sw)
rows = [{"word": w, "spam_rate": sp[w] / sum(spam), "ham_rate": hm[w] / (len(spam) - sum(spam)),
         "removed_by_us": w in cleaning.get_stopwords()} for w in sw if sp[w] + hm[w] >= 20]
pd.DataFrame(rows).assign(ratio=lambda d: d.spam_rate / (d.ham_rate + 1e-9)).sort_values("ratio", ascending=False).head(12)

## Task 5 - Save the cleaned dataset
The official way is `python -m src.run_pipeline` (it also writes `reports/results.json` for the report). Below we just verify the saved file.

In [ ]:
out = pd.read_csv(config.CLEAN_FILE)
print(out.shape); print(out.columns.tolist())
out.head()